In [2]:
import os
import sys
import math
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import scanpy as sc
import anndata as ad
import scvi
import session_info
import muon as mu

from scipy.io import mmread
from scipy.sparse import csr_matrix

sys.path.append('/nfs/casm/team215mg/users/od5/scripts/')
from single_cell_utils import *
from scjp_colors import *

In [3]:
## Set up work_dir
work_dir = '/lustre/scratch125/casm/teams/team215/users/od5/projects/celltherapy/ipsc/multiome/archr/tf_pilot/batch1/ipsc/'

## Fraction of cells per cell type each peak is accessible into:

In [4]:
## Read cell and peak info:
cell_info = pd.read_csv('/lustre/scratch125/casm/teams/team215/users/od5/projects/celltherapy/ipsc/multiome/gex_adata/tf_pilot/batch1/ipsc/leiden_v1/adata_multi_ipsc_perturbed_anno_v1_nuc_gex.csv', index_col=0)

## And add to adata
mdata = mu.read_h5mu('/lustre/scratch125/casm/teams/team215/users/od5/projects/celltherapy/ipsc/multiome/mdata/tf_pilot/batch1/ipsc/mdata_full.h5mu')
adata = mdata['atac']
adata.obs = cell_info.loc[adata.obs_names, :]

/software/conda/users/od5/scvi_24/lib/python3.9/site-packages/anndata/_core/anndata.py:430: FutureWarning: The dtype argument is deprecated and will be removed in late 2024.
  warnings.warn(
/software/conda/users/od5/scvi_24/lib/python3.9/site-packages/anndata/_core/anndata.py:430: FutureWarning: The dtype argument is deprecated and will be removed in late 2024.
  warnings.warn(


In [5]:
adata.obs['anno_v1_nuc_gex'].value_counts()

anno_v1_nuc_gex
Unperturbed    4061
TBX21           606
RUNX2           444
PRDM1           399
EBF1            363
CEBPD           319
Name: count, dtype: int64

In [6]:
adata.obs['perturbation'].value_counts()

perturbation
HOPX       1738
TCF7       1048
Control     856
FOXO3       576
TBX21       576
CEBPD       486
RUNX2       329
PRDM1       295
EBF1        282
CEBPA         6
Name: count, dtype: int64

In [7]:
def calculate_fraction(adata, cell_type):
    more_than_zero = np.array((adata[adata.obs['perturbation'] == cell_type, :].X > 0).mean(axis=0)).flatten()
    return pd.Series(more_than_zero , index=adata.var_names)

# Applying the function to each cell type and combining results
cell_types = adata.obs['perturbation'].unique()
results = {ct: calculate_fraction(adata, ct) for ct in cell_types}

# Convert the results dictionary to a DataFrame
results_df = pd.DataFrame(results)

In [8]:
results_df.to_csv((work_dir + 'archr_peaks_f_access_by_cellType.csv'))

In [9]:
results_df

,HOPX,TCF7,CEBPD,RUNX2,FOXO3,EBF1,TBX21,Control,PRDM1,CEBPA
chr1:827269-827769,0.042578,0.046756,0.041152,0.063830,0.045139,0.067376,0.062500,0.051402,0.027119,0.000000
chr1:845631-846131,0.019563,0.013359,0.037037,0.003040,0.013889,0.017730,0.020833,0.015187,0.023729,0.000000
chr1:857884-858384,0.003452,0.003817,0.004115,0.003040,0.008681,0.000000,0.005208,0.001168,0.003390,0.000000
chr1:869694-870194,0.039701,0.032443,0.039095,0.063830,0.041667,0.049645,0.060764,0.040888,0.023729,0.000000
chr1:904474-904974,0.055236,0.046756,0.043210,0.042553,0.055556,0.067376,0.046875,0.061916,0.047458,0.000000
...,...,...,...,...,...,...,...,...,...,...
chrX:155612044-155612544,0.010932,0.014313,0.014403,0.006079,0.015625,0.010638,0.026042,0.018692,0.013559,0.000000
chrX:155767370-155767870,0.040852,0.037214,0.053498,0.024316,0.032986,0.046099,0.036458,0.037383,0.047458,0.166667
chrX:155857465-155857965,0.006904,0.006679,0.002058,0.021277,0.008681,0.007092,0.003472,0.008178,0.013559,0.000000
chrX:155881055-155881555,0.061565,0.043893,0.063786,0.060790,0.053819,0.053191,0.059028,0.053738,0.054237,0.000000
